# **DirhamVision — UAE Currency Detection with YOLO26**

**Model:** YOLO26n · 2.4M params · 4ms inference  
**Classes:** `25_fils` · `50_fils` · `1_aed_coin` · `10_aed_bil`  
**Result:** 98.2% mAP@50

## Setting up the dataset:

In [ ]:
# ── SECTION 1: ENVIRONMENT SETUP ─────────────────────────────────

!nvidia-smi
!pip install ultralytics -q

In [ ]:
# ── SECTION 2: DATASET SETUP ─────────────────────────────────────

# 2.1 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2.2 Load dataset from Drive
import os, shutil

DATASET_ZIP = "/content/drive/MyDrive/new_dataset_full.zip"
DATASET_DIR = "/content/new_dataset_full"

!cp "{DATASET_ZIP}" /content/
!unzip -q /content/new_dataset_full.zip -d /content/

print(f"Images: {len(os.listdir(f'{DATASET_DIR}/images'))}")
print(f"Labels: {len(os.listdir(f'{DATASET_DIR}/labels'))}")

In [ ]:
# 2.3 [Label Studio only] Fix URL-encoded filenames
# Label Studio exports labels with %5C (URL-encoded backslash) in filenames.
# Skip this cell if your dataset was annotated with Roboflow or any other tool.

labels_dir = f"{DATASET_DIR}/labels"
fixed = 0

for label_file in os.listdir(labels_dir):
    if not label_file.endswith('.txt'):
        continue
    if '%5C' in label_file:
        clean_name = os.path.splitext(label_file.split('%5C')[-1])[0] + '.txt'
        os.rename(
            os.path.join(labels_dir, label_file),
            os.path.join(labels_dir, clean_name)
        )
        fixed += 1

print(f"Fixed {fixed} label filenames" if fixed else "No fixes needed — filenames already clean")

In [ ]:
# 2.4 Verify image-label pairs, split 85/15, write data.yaml
import random, shutil, yaml

images_dir = f"{DATASET_DIR}/images"
labels_dir = f"{DATASET_DIR}/labels"

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG', '.jpe', '.bmp'}

all_images = [f for f in os.listdir(images_dir)
              if os.path.splitext(f)[1] in IMAGE_EXTS]

matched, unmatched = [], []
for img in all_images:
    stem = os.path.splitext(img)[0]
    if os.path.exists(os.path.join(labels_dir, stem + '.txt')):
        matched.append(img)
    else:
        unmatched.append(img)

print(f"Matched pairs : {len(matched)}")
print(f"Unmatched     : {len(unmatched)}")
if unmatched:
    print("Unmatched files:", unmatched[:5])

random.seed(42)
random.shuffle(matched)
split = int(0.85 * len(matched))
train_imgs = matched[:split]
val_imgs   = matched[split:]
print(f"\nTrain: {len(train_imgs)} | Val: {len(val_imgs)}")

for split_name in ['train', 'val']:
    os.makedirs(f"{DATASET_DIR}/{split_name}/images", exist_ok=True)
    os.makedirs(f"{DATASET_DIR}/{split_name}/labels", exist_ok=True)

for img in train_imgs:
    stem = os.path.splitext(img)[0]
    shutil.copy(f"{images_dir}/{img}",        f"{DATASET_DIR}/train/images/{img}")
    shutil.copy(f"{labels_dir}/{stem}.txt",   f"{DATASET_DIR}/train/labels/{stem}.txt")

for img in val_imgs:
    stem = os.path.splitext(img)[0]
    shutil.copy(f"{images_dir}/{img}",        f"{DATASET_DIR}/val/images/{img}")
    shutil.copy(f"{labels_dir}/{stem}.txt",   f"{DATASET_DIR}/val/labels/{stem}.txt")

with open(f"{DATASET_DIR}/classes.txt") as f:
    classes = [l.strip() for l in f if l.strip()]

data_yaml = {
    'path'  : DATASET_DIR,
    'train' : 'train/images',
    'val'   : 'val/images',
    'nc'    : len(classes),
    'names' : classes
}
with open(f"{DATASET_DIR}/data.yaml", 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"\nClasses: {classes}")
print("data.yaml written ✓")

In [ ]:
# 2.5 Fix out-of-bounds label coordinates (Label Studio artifact)
# Clamps all coordinates to [0, 1] range. Safe to run regardless of annotation tool.

fixed = 0
for label_file in os.listdir(f"{DATASET_DIR}/train/labels"):
    if not label_file.endswith('.txt'):
        continue
    path = os.path.join(f"{DATASET_DIR}/train/labels", label_file)
    lines = open(path).readlines()
    new_lines, changed = [], False
    for line in lines:
        parts = line.strip().split()
        if len(parts) == 5:
            cls = parts[0]
            coords = [min(1.0, max(0.0, float(x))) for x in parts[1:]]
            new_line = f"{cls} {' '.join(f'{c:.6f}' for c in coords)}\n"
            if new_line != line:
                changed = True
            new_lines.append(new_line)
    if changed:
        open(path, 'w').writelines(new_lines)
        fixed += 1

print(f"Clamped {fixed} label files")

In [ ]:
# 2.6 Add background images (negative samples)
# Background images have no objects — teaches the model what NOT to fire on.
# Recommended: 2–5% of dataset size. We added 14 images (~2%).
# Upload plain photos of your detection surface with no objects in frame.

from google.colab import files as colab_files

print("Upload background images (no coins or bills in frame):")
uploaded = colab_files.upload()

train_images = f"{DATASET_DIR}/train/images"
train_labels = f"{DATASET_DIR}/train/labels"
added = 0

for filename, data in uploaded.items():
    if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        print(f"  Skipped {filename} — not an image")
        continue
    with open(os.path.join(train_images, filename), 'wb') as f:
        f.write(data)
    open(os.path.join(train_labels, os.path.splitext(filename)[0] + '.txt'), 'w').close()
    added += 1
    print(f"  Added: {filename}")

print(f"\n{added} background images added to training set")
print(f"Total train images now: {len(os.listdir(train_images))}")

## Training:



In [ ]:
# ── SECTION 3: TRAINING ───────────────────────────────────────────

from ultralytics import YOLO

DATA_YAML = f"{DATASET_DIR}/data.yaml"

In [ ]:
# 3.1 Baseline — yolo26s, no augmentation
# Used as a performance reference point across experiments.
# Achieves ~96% mAP@50. See FINDINGS.md for full comparison table and details.

model_baseline = YOLO("yolo26s.pt")
model_baseline.train(
    data=DATA_YAML,
    epochs=100, #Update epochs as needed
    imgsz=640,
    batch=16,
    patience=15,
    name="train_baseline_s",
    exist_ok=True
)

In [ ]:
# 3.2 Final model — yolo26n, 140 epochs, tuned augmentation
# This is the shipping model.
#
# Augmentation decisions:
#   mosaic=0.0   — off (default 1.0). Ultralytics docs recommend reducing
#                  mosaic for datasets under 1,000 images.
#   degrees=180  — full rotation (default 0.0). Fixes upside-down coin misses
#                  observed in video testing.
#   scale=0.2    — reduced (default 0.5). Mild scale variation for distance
#                  robustness without over-distorting small coin objects.
#   erasing=0.0  — off (default 0.4). Random erasing removes too much of small
#                  coin objects and degrades performance on this dataset size.

model_final = YOLO("yolo26n.pt")
model_final.train(
    data=DATA_YAML,
    epochs=140,
    imgsz=640,
    batch=16,
    patience=20,
    mosaic=0.0,
    degrees=180,
    scale=0.2,
    erasing=0.0,
    name="train_n_final",
    exist_ok=True
)

## Evaluation:



In [ ]:
# ── SECTION 4: EVALUATION ─────────────────────────────────────────

from ultralytics import YOLO
from IPython.display import Image, display

FINAL_WEIGHTS   = "/content/runs/detect/train_n_final/weights/best.pt"
BASELINE_WEIGHTS = "/content/runs/detect/train_baseline_s/weights/best.pt"
VAL_IMAGES      = f"{DATASET_DIR}/val/images"
VAL_LABELS      = f"{DATASET_DIR}/val/labels"

In [ ]:
# 4.1 Confusion matrices

print("Baseline — yolo26s, 50 epochs")
display(Image('/content/runs/detect/train_baseline_s/confusion_matrix_normalized.png', width=600))

print("Final — yolo26n, 140 epochs, tuned augmentation")
display(Image('/content/runs/detect/train_n_final/confusion_matrix_normalized.png', width=600))

In [ ]:
# 4.2 Visualize val set predictions — scroll through to inspect

model_eval = YOLO(FINAL_WEIGHTS)
model_eval.predict(
    source=VAL_IMAGES,
    conf=0.35,
    save=True,
    project="/content",
    name="val_predictions",
    exist_ok=True,
    verbose=False
)

import glob
pred_images = sorted(glob.glob('/content/val_predictions/*.jpg'))
print(f"{len(pred_images)} prediction images saved")

# Edit slice to browse different ranges
for img_path in pred_images[0:10]:
    display(Image(filename=img_path, height=400))
    print(os.path.basename(img_path), '\n')

In [ ]:
# 4.3 Error analysis — show only what the model got wrong
# Categorizes errors as: FP (hallucination) | FN (missed) | Wrong class
# Displays YOLO's own clean prediction boxes — no overlaid GT boxes.

import os, glob, numpy as np, cv2
from ultralytics import YOLO
from IPython.display import Image as IPImage, display

model_eval = YOLO(FINAL_WEIGHTS)

CLASS_NAMES = {0: '10_aed_bil', 1: '1_aed_coin', 2: '25_fils', 3: '50_fils'}

def iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter   = max(0, x2-x1) * max(0, y2-y1)
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    return inter / (a1 + a2 - inter + 1e-6)

def yolo_to_xyxy(line, w, h):
    _, cx, cy, bw, bh = map(float, line.split())
    return [int((cx-bw/2)*w), int((cy-bh/2)*h),
            int((cx+bw/2)*w), int((cy+bh/2)*h)]

results_all = model_eval.predict(
    source=VAL_IMAGES, conf=0.35, save=True,
    project="/content", name="error_check",
    exist_ok=True, verbose=False
)

errors = []
for result in results_all:
    img_path  = result.path
    stem      = os.path.splitext(os.path.basename(img_path))[0]
    label_path = os.path.join(VAL_LABELS, stem + '.txt')
    if not os.path.exists(label_path):
        continue

    img   = cv2.imread(img_path)
    h, w  = img.shape[:2]

    gt_boxes = [(int(l.split()[0]), yolo_to_xyxy(l.strip(), w, h))
                for l in open(label_path) if l.strip()]

    pred_boxes = [(int(b.cls[0]), b.xyxy[0].cpu().numpy().astype(int).tolist(),
                   float(b.conf[0])) for b in result.boxes]

    matched_gt, mistakes = set(), []
    for pcls, pbox, pconf in pred_boxes:
        best_iou, best_gi = 0.15, -1
        for gi, (gcls, gbox) in enumerate(gt_boxes):
            if gi in matched_gt: continue
            s = iou(pbox, gbox)
            if s > best_iou:
                best_iou, best_gi = s, gi
        if best_gi == -1:
            mistakes.append(('FP', pcls, None, pconf))
        else:
            matched_gt.add(best_gi)
            if pcls != gt_boxes[best_gi][0]:
                mistakes.append(('WRONG_CLASS', pcls, gt_boxes[best_gi][0], pconf))

    for gi, (gcls, _) in enumerate(gt_boxes):
        if gi not in matched_gt:
            mistakes.append(('FN', None, gcls, 0))

    if mistakes:
        errors.append((img_path, mistakes))

fp = sum(1 for _, ms in errors for m in ms if m[0] == 'FP')
fn = sum(1 for _, ms in errors for m in ms if m[0] == 'FN')
wc = sum(1 for _, ms in errors for m in ms if m[0] == 'WRONG_CLASS')
print(f"Error images : {len(errors)} / {len(results_all)}")
print(f"FP           : {fp}")
print(f"FN (missed)  : {fn}")
print(f"Wrong class  : {wc}\n")

bill_errors  = [(p, ms) for p, ms in errors if any(m[1]==0 or m[2]==0 for m in ms)]
other_errors = [(p, ms) for p, ms in errors if not any(m[1]==0 or m[2]==0 for m in ms)]

def print_errors(error_list, limit=15):
    for img_path, mistakes in error_list[:limit]:
        print(os.path.basename(img_path))
        for m in mistakes:
            if   m[0] == 'FP':         print(f"  FP:          {CLASS_NAMES[m[1]]} conf={m[3]:.2f}")
            elif m[0] == 'FN':         print(f"  MISSED:      {CLASS_NAMES[m[2]]}")
            else:                      print(f"  WRONG CLASS: predicted {CLASS_NAMES[m[1]]} | GT {CLASS_NAMES[m[2]]}")
        saved = f"/content/error_check/{os.path.basename(img_path)}"
        if os.path.exists(saved):
            display(IPImage(filename=saved, height=350))
        print()

print(f"=== 10_aed_bil errors ({len(bill_errors)}) ===\n")
print_errors(bill_errors)

print(f"=== Other class errors ({len(other_errors)}) ===\n")
print_errors(other_errors)

In [ ]:
# ── SECTION 5: INFERENCE ON VIDEO ────────────────────────────────

import cv2
from ultralytics import YOLO

DENOMINATION_VALUES = {
    '25_fils': 0.25, '50_fils': 0.50,
    '1_aed_coin': 1.00, '10_aed_bil': 10.00
}
CLASS_COLORS = {
    '25_fils':    (0, 215, 255),
    '50_fils':    (0, 165, 255),
    '1_aed_coin': (0, 255,   0),
    '10_aed_bil': (255, 0,  255)
}

def process_video(video_path, model, output_tag, conf=0.35):
    cap    = cv2.VideoCapture(video_path)
    fps    = int(cap.get(cv2.CAP_PROP_FPS)) or 30
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    base     = os.path.splitext(os.path.basename(video_path))[0]
    out_path = f"/content/{base}__{output_tag}.mp4"
    writer   = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    print(f"  {width}x{height} @ {fps}fps — {total} frames")

    for frame_n in range(total):
        ret, frame = cap.read()
        if not ret:
            break

        results   = model(frame, conf=conf, verbose=False)[0]
        total_aed = 0.0
        detected  = {}

        for box in results.boxes:
            cls_name = model.names[int(box.cls[0])]
            score    = float(box.conf[0])
            total_aed += DENOMINATION_VALUES.get(cls_name, 0)
            detected[cls_name] = detected.get(cls_name, 0) + 1

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            color = CLASS_COLORS.get(cls_name, (255, 255, 255))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label = f"{cls_name} {score:.2f}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
            cv2.rectangle(frame, (x1, y1-th-8), (x1+tw+6, y1), color, -1)
            cv2.putText(frame, label, (x1+3, y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,0), 1)

        panel_h = 60 + len(detected) * 28
        overlay = frame.copy()
        cv2.rectangle(overlay, (8, 8), (320, panel_h), (20, 20, 20), -1)
        cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)
        cv2.putText(frame, f"Total: {total_aed:.2f} AED",
                    (16, 44), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 120), 2)
        y = 70
        for cls_name, count in detected.items():
            val   = DENOMINATION_VALUES.get(cls_name, 0)
            color = CLASS_COLORS.get(cls_name, (255, 255, 255))
            cv2.putText(frame, f"{cls_name} x{count}  ({val*count:.2f} AED)",
                        (16, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            y += 28

        writer.write(frame)
        if frame_n % 100 == 0:
            print(f"  {frame_n}/{total}", end='\r')

    cap.release()
    writer.release()
    print(f"  Saved → {out_path}")
    return out_path

In [ ]:
# 5.1 Process a single video

from google.colab import files as colab_files

MODEL_PATH     = FINAL_WEIGHTS
CONF_THRESHOLD = 0.35

model_infer = YOLO(MODEL_PATH)

print("Upload a video:")
uploaded    = colab_files.upload()
video_name  = list(uploaded.keys())[0]
video_path  = f"/content/{video_name}"

output = process_video(video_path, model_infer, output_tag="dirhamvision", conf=CONF_THRESHOLD)
colab_files.download(output)

In [ ]:
# 5.2 Batch — process multiple videos, download all outputs

print("Upload videos (select multiple):")
uploaded = colab_files.upload()

model_infer = YOLO(FINAL_WEIGHTS)
outputs     = []

for video_name in uploaded.keys():
    if not video_name.endswith('.mp4'):
        continue
    video_path = f"/content/{video_name}"
    print(f"\nProcessing: {video_name}")
    out = process_video(video_path, model_infer, output_tag="dirhamvision", conf=0.35)
    outputs.append(out)

print(f"\nAll done — {len(outputs)} videos processed")
for out in outputs:
    colab_files.download(out)

## Saving Models:



In [ ]:
# ── SECTION 6: EXPORT MODEL WEIGHTS ──────────────────────────────

import shutil

shutil.make_archive('/content/train_n_final', 'zip', '/content/runs/detect/train_n_final')
shutil.make_archive('/content/train_baseline_s', 'zip', '/content/runs/detect/train_baseline_s')

from google.colab import files as colab_files
colab_files.download('/content/train_n_final.zip')
colab_files.download('/content/train_baseline_s.zip')

print("Downloaded: train_n_final.zip (shipping model)")
print("Downloaded: train_baseline_s.zip (baseline for comparison)")